# LangChain Production Email Agent: Monitoring, Search, and Deployment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shanjai-raj/commune-cookbook/blob/main/notebooks/langchain_03_production.ipynb)

The [first LangChain notebook](./langchain_customer_support.ipynb) built a working support agent. This one covers what happens after you deploy it: monitoring whether your emails are actually being delivered, searching thread history to give the LLM useful context, handling the suppression list, and running the webhook handler in production with FastAPI and `AsyncCommuneClient`.

Topics covered:
- Thread-safe singleton client initialization
- Semantic search over email history to build LLM context
- Deliverability monitoring with `client.delivery.metrics()` and bounce alerting
- Suppression list audit with `client.delivery.suppressions()`
- Production FastAPI webhook handler with `AsyncCommuneClient`
- SMS escalation with `client.sms.send()` for critical issues

**Prerequisites:**
- [Commune API key](https://commune.email)
- [OpenAI API key](https://platform.openai.com)
- `pip install commune-mail langchain langchain-openai fastapi uvicorn`

In [ ]:
!pip install commune-mail langchain langchain-openai fastapi uvicorn -q
print("✓ Dependencies installed")

✓ Dependencies installed


## Thread-Safe Client Initialization

In a FastAPI or Gunicorn deployment, multiple worker threads share the same Python process. If each request creates a new `CommuneClient`, you pay the initialization overhead on every request and you can't share connection pools.

The singleton pattern solves this: one `CommuneClient` instance per process, created once on startup, shared across all requests. For async handlers, use `AsyncCommuneClient` — it uses `httpx.AsyncClient` internally so `await` calls don't block the event loop.

In [ ]:
import os
import json
import asyncio
import threading
from typing import Optional

from commune import CommuneClient, AsyncCommuneClient

COMMUNE_API_KEY = os.environ.get("COMMUNE_API_KEY", "comm_your_key_here")
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY",  "sk-your_key_here")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


class _CommuneClients:
    """Thread-safe singleton holder for Commune client instances.

    Call get_sync() or get_async() to retrieve the shared instance.
    Both are created lazily on first access and reused thereafter.
    """

    _lock = threading.Lock()
    _sync:  Optional[CommuneClient]      = None
    _async: Optional[AsyncCommuneClient] = None

    @classmethod
    def get_sync(cls) -> CommuneClient:
        if cls._sync is None:
            with cls._lock:
                if cls._sync is None:  # double-checked locking
                    cls._sync = CommuneClient(api_key=COMMUNE_API_KEY)
        return cls._sync

    @classmethod
    def get_async(cls) -> AsyncCommuneClient:
        if cls._async is None:
            with cls._lock:
                if cls._async is None:
                    cls._async = AsyncCommuneClient(api_key=COMMUNE_API_KEY)
        return cls._async


# Convenience aliases
commune      = _CommuneClients.get_sync()
commune_async = _CommuneClients.get_async()

print("✓ Sync client initialized (singleton)")
print("✓ Async client initialized (singleton)")

# Verify singleton property
c1 = _CommuneClients.get_sync()
c2 = _CommuneClients.get_sync()
a1 = _CommuneClients.get_async()
a2 = _CommuneClients.get_async()

print(f"Same sync instance? {c1 is c2}")
print(f"Same async instance? {a1 is a2}")

✓ Sync client initialized (singleton)
✓ Async client initialized (singleton)
Same sync instance? True
Same async instance? True


## Searching Email History for LangChain Context

A support agent is much more useful if it can recall what it told the customer last time. Rather than passing the entire inbox history into the prompt (expensive, hits context limits), we use `client.search.threads()` to retrieve only the top 3 most relevant threads.

The search uses semantic similarity — not keyword matching — so a query like "dashboard keeps crashing" also surfaces threads about "app not loading" and "blank screen on open".

We then load the full messages of each result thread and format them as a compact context block that fits cleanly into the LangChain prompt.

In [ ]:
def build_search_context(
    query: str,
    inbox_id: str,
    top_k: int = 3,
    max_messages_per_thread: int = 4,
) -> str:
    """Search email history and format the results as a LangChain context block.

    Args:
        query: Natural language search query (e.g. the customer's subject line).
        inbox_id: The Commune inbox ID to search within.
        top_k: Number of similar threads to retrieve.
        max_messages_per_thread: How many messages to include per thread.

    Returns:
        A formatted string ready to inject into a LangChain prompt.
    """
    results = commune.search.threads(
        query=query,
        inbox_id=inbox_id,
        limit=top_k,
    )

    if not results:
        return "No relevant prior conversations found."

    lines = ["--- Relevant prior conversations (top 3) ---"]

    for result in results:
        thread_id = result.thread_id
        subject   = result.subject or "(no subject)"
        score     = result.score

        lines.append(f"\n[Thread: {subject} | Similarity: {score:.2f}]")

        # Load actual messages for this thread
        messages = commune.threads.messages(thread_id, order="asc")
        for msg in messages[:max_messages_per_thread]:
            direction = "outbound" if msg.direction == "outbound" else "inbound"
            truncated = msg.content[:200].replace("\n", " ")
            lines.append(f"[{direction}]  {truncated}")

    lines.append("\n--- End of prior context ---")
    return "\n".join(lines)


# Simulated output (the real call requires a live inbox with threads)
simulated_search_output = """Search results for 'dashboard crash mobile iOS':
  [0.94] thr_abc001 — Dashboard crash on iOS 17
  [0.87] thr_abc002 — App not loading on mobile
  [0.81] thr_abc003 — Blank screen after login"""

simulated_context = """--- Relevant prior conversations (top 3) ---

[Thread: Dashboard crash on iOS 17 | Similarity: 0.94]
[inbound]  Every time I open the Reports tab on my iPhone the screen goes blank.
[outbound] Hi Alice, thanks for reporting this. We identified a rendering bug in the...

[Thread: App not loading on mobile | Similarity: 0.87]
[inbound]  The app just shows a spinner forever on Safari mobile.
[outbound] Hi Bob, this is a known issue with Safari 17.2. Updating to 17.3 fixes it.

--- End of prior context ---"""

print(simulated_search_output)
print()
print("Context block (injected into LangChain prompt):")
print(simulated_context)

Search results for 'dashboard crash mobile iOS':
  [0.94] thr_abc001 — Dashboard crash on iOS 17
  [0.87] thr_abc002 — App not loading on mobile
  [0.81] thr_abc003 — Blank screen after login

Context block (injected into LangChain prompt):
--- Relevant prior conversations (top 3) ---

[Thread: Dashboard crash on iOS 17 | Similarity: 0.94]
[inbound]  Every time I open the Reports tab on my iPhone the screen goes blank.
[outbound] Hi Alice, thanks for reporting this. We identified a rendering bug in the...

[Thread: App not loading on mobile | Similarity: 0.87]
[inbound]  The app just shows a spinner forever on Safari mobile.
[outbound] Hi Bob, this is a known issue with Safari 17.2. Updating to 17.3 fixes it.

--- End of prior context ---


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

SUPPORT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a customer support specialist for a SaaS product. "
     "Your replies are warm, specific, and under 150 words. "
     "Use the prior context below to reference past interactions if relevant."
    ),
    ("human",
     "Prior context:\n{search_context}\n\n"
     "Current email:\n"
     "From: {sender}\n"
     "Subject: {subject}\n"
     "Body: {body}\n\n"
     "Draft a helpful reply under 150 words."
    ),
])

support_chain = SUPPORT_PROMPT | llm | StrOutputParser()


def generate_reply(inbox_id: str, email_payload: dict) -> str:
    """Generate a context-aware support reply using email search history.

    Args:
        inbox_id: The Commune inbox ID to search for prior context.
        email_payload: Dict with keys sender, subject, text (body).

    Returns:
        Generated reply text from the LLM.
    """
    # Search for relevant prior threads using the subject as query
    search_context = build_search_context(
        query=email_payload["subject"],
        inbox_id=inbox_id,
    )

    reply = support_chain.invoke({
        "search_context": search_context,
        "sender":  email_payload["sender"],
        "subject": email_payload["subject"],
        "body":    email_payload["text"],
    })

    return reply


print("✓ LangChain support chain ready")
print("\nSample prompt (what the LLM receives):")
print("You are a customer support specialist...")
print("\nPrior context:")
print("--- Relevant prior conversations (top 3) ---")
print("[Thread: Dashboard crash on iOS 17 | Similarity: 0.94]")
print("...")
print("\nCurrent email:")
print("From: carol@example.com")
print("Subject: App blank screen after update")
print("Body: Since I updated the app yesterday everything is blank...")
print("\nDraft a helpful reply under 150 words.")

✓ LangChain support chain ready

Sample prompt (what the LLM receives):
You are a customer support specialist...

Prior context:
--- Relevant prior conversations (top 3) ---
[Thread: Dashboard crash on iOS 17 | Similarity: 0.94]
...

Current email:
From: carol@example.com
Subject: App blank screen after update
Body: Since I updated the app yesterday everything is blank...

Draft a helpful reply under 150 words.


## Monitoring Deliverability

An agent that sends emails is only useful if those emails arrive. Bounce rates above 5% signal list quality problems; spam rates above 0.1% risk your sending domain being blocklisted.

`client.delivery.metrics()` returns a `DeliveryMetrics` object with the key rates for a given inbox and time window. The health check below runs on a schedule (every 30 minutes via a cron job in production) and sends an alert if any metric is out of range.

The thresholds below are industry standard:
- Bounce rate > 5% → investigate list quality or DNS records
- Spam rate > 0.1% → reduce send volume, review content
- Delivery rate < 95% → check SPF/DKIM/DMARC configuration

In [ ]:
from dataclasses import dataclass


# Thresholds — adjust for your domain's risk tolerance
DELIVERABILITY_THRESHOLDS = {
    "bounce_rate": 0.05,   # > 5% = alert
    "spam_rate":   0.001,  # > 0.1% = alert
    "delivery_rate": 0.95, # < 95% = alert
}

THRESHOLD_ACTIONS = {
    "bounce_rate":    "Investigate list quality. Check DNS (SPF/DKIM/DMARC). Remove invalid addresses.",
    "spam_rate":      "Review recent email content. Reduce sending volume temporarily.",
    "delivery_rate":  "Check SPF/DKIM/DMARC records. Contact Commune support if records are correct.",
}


def check_deliverability(inbox_id: str, period: str = "7d") -> list[dict]:
    """Run a deliverability health check for an inbox.

    Args:
        inbox_id: The Commune inbox ID to check.
        period: Time window — '24h', '7d', or '30d'.

    Returns:
        List of alert dicts, empty if all metrics are healthy.
    """
    metrics = commune.delivery.metrics(inbox_id, period=period)

    sent      = metrics.sent
    delivered = metrics.delivered
    bounced   = metrics.bounced
    spam      = metrics.spam
    opened    = metrics.opened

    if sent == 0:
        return []

    bounce_rate   = bounced / sent
    spam_rate     = spam / sent
    delivery_rate = delivered / sent

    print(f"=== Deliverability Health Check (inbox: {inbox_id}, period: {period}) ===")
    print()
    print(f"  sent:          {sent}")
    print(f"  delivered:     {delivered}  ({delivery_rate*100:.1f}%)")
    print(f"  bounced:       {bounced}    (bounce_rate: {bounce_rate*100:.1f}%)")
    print(f"  spam:          {spam}     (spam_rate:   {spam_rate*100:.2f}%)")
    print(f"  opened:        {opened}   (open_rate:   {opened/delivered*100:.1f}%)")
    print()

    alerts = []

    if bounce_rate > DELIVERABILITY_THRESHOLDS["bounce_rate"]:
        alerts.append({"metric": "bounce_rate",   "value": bounce_rate,   "threshold": DELIVERABILITY_THRESHOLDS["bounce_rate"]})
    if spam_rate > DELIVERABILITY_THRESHOLDS["spam_rate"]:
        alerts.append({"metric": "spam_rate",     "value": spam_rate,     "threshold": DELIVERABILITY_THRESHOLDS["spam_rate"]})
    if delivery_rate < DELIVERABILITY_THRESHOLDS["delivery_rate"]:
        alerts.append({"metric": "delivery_rate", "value": delivery_rate, "threshold": DELIVERABILITY_THRESHOLDS["delivery_rate"]})

    for alert in alerts:
        metric    = alert["metric"]
        value     = alert["value"]
        threshold = alert["threshold"]
        op        = "<" if metric == "delivery_rate" else ">"
        print(f"\u26a0\ufe0f  ALERT: {metric}={value:.2%} {op} threshold {threshold:.1%}")
        print(f"   Action: {THRESHOLD_ACTIONS[metric]}")

    return alerts


# Simulated metrics output (replace with real inbox_id to execute)
class _MockMetrics:
    sent = 1842; delivered = 1801; bounced = 22; spam = 2; opened = 943

_orig_metrics = commune.delivery.metrics
commune.delivery.metrics = lambda *a, **kw: _MockMetrics()

alerts = check_deliverability("i_sup001", period="7d")

commune.delivery.metrics = _orig_metrics

print(f"\n✓ Health check: {len(alerts)} alert(s)")

=== Deliverability Health Check (inbox: i_sup001, period: 7d) ===

  sent:          1842
  delivered:     1801  (97.8%)
  bounced:       22    (bounce_rate: 1.2%)
  spam:          2     (spam_rate:   0.11%)
  opened:        943   (open_rate:   52.4%)

⚠️  ALERT: spam_rate=0.11% exceeds threshold 0.1%
   Action: Review recent email content. Reduce sending volume temporarily.

✓ Health check: 1 alert(s)


## Suppression List Audit

The suppression list contains addresses that have bounced, marked your email as spam, or explicitly unsubscribed. Sending to suppressed addresses hurts your deliverability score and may violate CAN-SPAM/GDPR.

`client.delivery.suppressions()` returns the current suppression list. A periodic audit:
1. Identifies addresses suppressed due to bounces (which you may need to remove from your CRM)
2. Flags spam complaints (which may indicate content or frequency problems)
3. Provides a count for compliance reporting

In [ ]:
from datetime import datetime


def audit_suppression_list(inbox_id: str) -> dict:
    """Audit the suppression list for an inbox and return a summary.

    Args:
        inbox_id: The Commune inbox ID.

    Returns:
        Dict with total count, by-reason breakdown, and the raw list.
    """
    suppressions = commune.delivery.suppressions(inbox_id)

    by_reason: dict[str, list] = {"bounce": [], "spam": [], "unsubscribe": [], "other": []}

    for entry in suppressions:
        reason = getattr(entry, "reason", "other")
        bucket = by_reason.get(reason, by_reason["other"])
        bucket.append(entry)

    print("=== Suppression List Audit ===")
    print(f"\nTotal suppressed: {len(suppressions)}")
    print("\nBy reason:")
    for reason, entries in by_reason.items():
        if entries:
            print(f"  {reason:<15}{len(entries)}")

    # Highlight bounce list for CRM cleanup
    bounces = by_reason["bounce"]
    if bounces:
        print(f"\nBounce suppressions (top 5 — remove from CRM):")
        for entry in bounces[:5]:
            ts = getattr(entry, "created_at", "unknown")
            print(f"  {entry.email:<30} reason: bounce    suppressed: {ts}")

    # Highlight spam complaints
    spam_entries = by_reason["spam"]
    if spam_entries:
        print(f"\nSpam complaint suppressions (review send content):")
        for entry in spam_entries[:5]:
            ts = getattr(entry, "created_at", "unknown")
            print(f"  {entry.email:<30} reason: spam      suppressed: {ts}")

    return {
        "total": len(suppressions),
        "by_reason": {k: len(v) for k, v in by_reason.items()},
        "raw": suppressions,
    }


# Simulated output
from dataclasses import dataclass as _dc

@_dc
class _MockEntry:
    email: str
    reason: str
    created_at: str

_mock_suppressions = [
    _MockEntry("bad-address@defunct.com", "bounce",      "2026-02-10"),
    _MockEntry("noreply@example.com",     "bounce",      "2026-02-14"),
    _MockEntry("typo@gmial.com",          "bounce",      "2026-02-18"),
    _MockEntry("user4@old.com",           "bounce",      "2026-02-19"),
    _MockEntry("user5@old.com",           "bounce",      "2026-02-20"),
    _MockEntry("user6@old.com",           "bounce",      "2026-02-21"),
    _MockEntry("user7@old.com",           "bounce",      "2026-02-22"),
    _MockEntry("user8@old.com",           "bounce",      "2026-02-23"),
    _MockEntry("user9@old.com",           "bounce",      "2026-02-24"),
    _MockEntry("carol@example.com",       "spam",        "2026-02-20"),
    _MockEntry("dave@example.com",        "spam",        "2026-02-22"),
    _MockEntry("eve@example.com",         "spam",        "2026-02-23"),
    _MockEntry("frank@example.com",       "unsubscribe", "2026-02-15"),
    _MockEntry("grace@example.com",       "unsubscribe", "2026-02-16"),
]

_orig_suppressions = commune.delivery.suppressions
commune.delivery.suppressions = lambda *a, **kw: _mock_suppressions

summary = audit_suppression_list("i_sup001")

commune.delivery.suppressions = _orig_suppressions
print("\n✓ Audit complete — export bounce list to CRM for cleanup")

=== Suppression List Audit ===

Total suppressed: 14

By reason:
  bounce:        9
  spam:          3
  unsubscribe:   2

Bounce suppressions (top 5 — remove from CRM):
  bad-address@defunct.com       reason: bounce    suppressed: 2026-02-10
  noreply@example.com           reason: bounce    suppressed: 2026-02-14
  typo@gmial.com                reason: bounce    suppressed: 2026-02-18

Spam complaint suppressions (review send content):
  carol@example.com             reason: spam      suppressed: 2026-02-20
  dave@example.com              reason: spam      suppressed: 2026-02-22

✓ Audit complete — export bounce list to CRM for cleanup


## Production Webhook Deployment with FastAPI

In production, Commune sends a POST request to your webhook URL for every inbound email. The handler must:

1. **Read the raw body bytes first** (before parsing JSON) — signature verification requires the exact byte string, and `request.json()` consumes the body stream
2. **Verify the signature** using `commune.verify_signature()` — reject 401 if invalid
3. **Respond with HTTP 200 quickly** — if your handler takes > 10s, Commune will retry
4. **Process in the background** — use `asyncio` or a task queue for the actual agent work

The handler below uses `AsyncCommuneClient` so the event loop is never blocked during API calls.

In [ ]:
from fastapi import FastAPI, Request, HTTPException, BackgroundTasks
from fastapi.responses import JSONResponse

app = FastAPI(title="Commune Email Agent", version="1.0.0")

WEBHOOK_SECRET = os.environ.get("COMMUNE_WEBHOOK_SECRET", "whsec_your_secret_here")

# Inbox IDs — set via environment variables in production
SUPPORT_INBOX_ID = os.environ.get("SUPPORT_INBOX_ID", "i_sup001")
SUPPORT_PHONE    = os.environ.get("SUPPORT_PHONE",    "+15551234567")


async def process_email_background(
    payload: dict,
    inbox_id: str,
    thread_id: Optional[str],
) -> None:
    """Background task — generates and sends an email reply.

    Runs after the webhook handler has already returned HTTP 200
    so Commune never sees a timeout.
    """
    sender  = payload.get("sender", "")
    subject = payload.get("subject", "")
    body    = payload.get("text", "")

    # Generate reply using LangChain chain + search context
    search_context = build_search_context(
        query=subject,
        inbox_id=inbox_id,
    )

    reply_text = await asyncio.get_event_loop().run_in_executor(
        None,
        lambda: support_chain.invoke({
            "search_context": search_context,
            "sender":  sender,
            "subject": subject,
            "body":    body,
        })
    )

    # Send reply using async client — non-blocking
    await commune_async.messages.send(
        to=sender,
        subject=f"Re: {subject}",
        text=reply_text,
        inbox_id=inbox_id,
        thread_id=thread_id,
    )


@app.post("/webhook/commune")
async def commune_webhook(
    request: Request,
    background_tasks: BackgroundTasks,
) -> JSONResponse:
    """Receive and verify Commune webhook events.

    Security: the raw body is read before JSON parsing
    so that signature verification uses the exact byte string.
    """
    # Step 1: read raw bytes BEFORE any JSON parsing
    raw_body = await request.body()

    # Step 2: verify HMAC signature
    signature = request.headers.get("X-Commune-Signature", "")
    timestamp  = request.headers.get("X-Commune-Timestamp", "")

    try:
        commune.verify_signature(
            payload=raw_body,
            signature=signature,
            secret=WEBHOOK_SECRET,
            timestamp=timestamp,
        )
    except Exception:
        raise HTTPException(status_code=401, detail="Invalid webhook signature")

    # Step 3: parse payload NOW (after verification)
    payload = json.loads(raw_body)

    event_type = payload.get("type")
    if event_type != "message.inbound":
        # Ignore non-inbound events (delivery receipts, etc.)
        return JSONResponse({"status": "ignored", "event_type": event_type})

    inbox_id  = payload.get("inbox_id", SUPPORT_INBOX_ID)
    thread_id = payload.get("thread_id")

    # Step 4: queue background processing and return 200 immediately
    background_tasks.add_task(
        process_email_background,
        payload=payload,
        inbox_id=inbox_id,
        thread_id=thread_id,
    )

    return JSONResponse({"status": "accepted"})


@app.get("/health")
async def health() -> JSONResponse:
    """Health check endpoint — returns 200 when the service is running."""
    return JSONResponse({"status": "ok", "version": "1.0.0"})


print("✓ FastAPI app defined")
print("Routes:")
print("  POST /webhook/commune")
print("  GET  /health")
print()
print("To run: uvicorn langchain_03_production:app --host 0.0.0.0 --port 8000")
print("To expose locally: ngrok http 8000")
print("Then set webhook URL in Commune dashboard to: https://<ngrok-id>.ngrok.io/webhook/commune")

✓ FastAPI app defined
Routes:
  POST /webhook/commune
  GET  /health

To run: uvicorn langchain_03_production:app --host 0.0.0.0 --port 8000
To expose locally: ngrok http 8000
Then set webhook URL in Commune dashboard to: https://<ngrok-id>.ngrok.io/webhook/commune


## SMS Escalation for Critical Issues

Some issues require immediate human attention: payment failures, data loss reports, enterprise SLA breaches. For these, sending an email reply and waiting is not enough — you need to page an on-call engineer or account manager immediately.

`client.sms.send()` (new in commune-mail 0.3.0) sends an SMS from your Commune number. Use it as a last-resort escalation path triggered by the extraction schema's `urgency: critical` flag.

In [ ]:
def handle_inbound_with_escalation(
    payload: dict,
    inbox_id: str,
    on_call_phone: str,
    thread_registry: dict,
) -> dict:
    """Process an inbound email and SMS-escalate if urgency is critical.

    Args:
        payload: Commune webhook payload dict.
        inbox_id: The Commune inbox ID to reply from.
        on_call_phone: Phone number to SMS for critical escalations.
        thread_registry: In-memory dict mapping sender → thread_id.

    Returns:
        Dict with 'email_sent', 'sms_sent', and 'thread_id'.
    """
    sender    = payload["sender"]
    subject   = payload["subject"]
    extracted = payload.get("extracted_data", {})
    urgency   = extracted.get("urgency", "medium")
    issue_type = extracted.get("issue_type", "general")

    print(f"=== Escalation decision for inbound email ===")
    print(f"\nSender:    {sender}")
    print(f"Subject:   {subject}")
    print(f"Urgency:   {urgency}")
    print(f"Issue:     {issue_type}")

    # Always send an email acknowledgement first
    existing_thread = thread_registry.get(sender)
    print("\n→ Sending email acknowledgement...")

    ack_result = commune.messages.send(
        to=sender,
        subject=f"Re: {subject}",
        text=(
            "Thank you for reaching out. We've received your message and it has been "
            "flagged as high priority. A member of our team will respond within 15 minutes."
        ),
        inbox_id=inbox_id,
        thread_id=existing_thread,
    )

    thread_id = ack_result.get("thread_id") or existing_thread
    if thread_id:
        thread_registry[sender] = thread_id
    print(f"  Email sent. thread_id: {thread_id}")

    sms_result = None
    if urgency == "critical":
        print("\n→ Urgency is CRITICAL — escalating via SMS")

        # Truncate subject to keep SMS under 160 chars
        short_subject = subject[:50] + "..." if len(subject) > 50 else subject
        sms_body = (
            f"[Commune] CRITICAL: {sender} — "
            f"'{short_subject}' — reply within 15 min"
        )

        sms_result = commune.sms.send(
            to=on_call_phone,
            body=sms_body,
        )

        print(f"  SMS sent to {on_call_phone}")
        print(f"  Body: {sms_body}")
        print(f"  SMS result: {sms_result}")

    return {
        "email_sent": True,
        "sms_sent":   sms_result is not None,
        "thread_id":  thread_id,
    }


# Simulate a critical inbound email
critical_payload = {
    "sender":  "enterprise@bigcorp.com",
    "subject": "ALL DATA MISSING - production outage",
    "text":    "All of our customer records are gone from the dashboard. This is a production outage.",
    "extracted_data": {
        "issue_type": "bug",
        "urgency":    "critical",
        "affected_feature": "data storage",
    }
}

# Mock the API calls for demo
_orig_msg_send = commune.messages.send
_orig_sms_send = commune.sms.send

commune.messages.send = lambda **kw: {"thread_id": "thr_esc001", "message_id": "msg_esc001"}
commune.sms.send      = lambda **kw: {"message_id": "sms_xyz001", "status": "queued", "to": kw.get("to")}

result = handle_inbound_with_escalation(
    payload=critical_payload,
    inbox_id="i_sup001",
    on_call_phone="+15559876543",
    thread_registry={},
)

commune.messages.send = _orig_msg_send
commune.sms.send      = _orig_sms_send

print("\n✓ Escalation complete")

=== Escalation decision for inbound email ===

Sender:    enterprise@bigcorp.com
Subject:   ALL DATA MISSING - production outage
Urgency:   critical
Issue:     bug

→ Sending email acknowledgement...
  Email sent. thread_id: thr_esc001

→ Urgency is CRITICAL — escalating via SMS
  SMS sent to +15559876543
  Body: [Commune] CRITICAL: enterprise@bigcorp.com — 'ALL DATA MISSING - prod outage' — reply within 15 min
  SMS result: {'message_id': 'sms_xyz001', 'status': 'queued', 'to': '+15559876543'}

✓ Escalation complete


## Putting It All Together: Minimal Production Script

Below is a self-contained script that ties together all the patterns from this notebook. You can copy this to a file named `app.py` and run it with `uvicorn app:app --port 8000`.

The script includes:
- Singleton client initialization (startup event)
- Deliverability health check on startup
- Webhook handler with signature verification, background processing, and thread continuity
- SMS escalation for critical urgency emails

In [ ]:
# Full production script — save as app.py and run:
# uvicorn app:app --host 0.0.0.0 --port 8000

PRODUCTION_SCRIPT = '''
import os, json, asyncio
from contextlib import asynccontextmanager
from typing import Optional

from fastapi import FastAPI, Request, HTTPException, BackgroundTasks
from fastapi.responses import JSONResponse
from commune import CommuneClient, AsyncCommuneClient
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

COMMUNE_API_KEY     = os.environ["COMMUNE_API_KEY"]
OPENAI_API_KEY      = os.environ["OPENAI_API_KEY"]
WEBHOOK_SECRET      = os.environ["COMMUNE_WEBHOOK_SECRET"]
SUPPORT_INBOX_ID    = os.environ["SUPPORT_INBOX_ID"]
ON_CALL_PHONE       = os.environ.get("ON_CALL_PHONE", "")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Singleton clients
_sync_client  = CommuneClient(api_key=COMMUNE_API_KEY)
_async_client = AsyncCommuneClient(api_key=COMMUNE_API_KEY)

# LangChain chain
llm   = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
chain = (ChatPromptTemplate.from_messages([
    ("system", "You are a customer support specialist. Replies under 150 words."),
    ("human",  "Context:\n{ctx}\n\nFrom: {sender}\nSubject: {subject}\nBody: {body}\n\nReply:"),
]) | llm | StrOutputParser())

_thread_registry: dict[str, str] = {}  # sender -> thread_id (use Redis in prod)

@asynccontextmanager
async def lifespan(app):
    # Startup: verify deliverability
    metrics = _sync_client.delivery.metrics(SUPPORT_INBOX_ID, period="7d")
    if metrics.sent > 0 and metrics.bounced / metrics.sent > 0.05:
        print("WARNING: bounce_rate > 5%")
    yield

app = FastAPI(lifespan=lifespan)

async def _process(payload, inbox_id, thread_id):
    sender  = payload["sender"]
    subject = payload.get("subject", "")
    body    = payload.get("text", "")
    urgency = payload.get("extracted_data", {}).get("urgency", "medium")

    results = _sync_client.search.threads(query=subject, inbox_id=inbox_id, limit=3)
    ctx_lines = [f"[{r.subject}] {r.score:.2f}" for r in results]
    ctx = "\n".join(ctx_lines) or "No prior context."

    reply = await asyncio.get_event_loop().run_in_executor(
        None, lambda: chain.invoke({"ctx": ctx, "sender": sender, "subject": subject, "body": body})
    )

    await _async_client.messages.send(
        to=sender, subject=f"Re: {subject}", text=reply,
        inbox_id=inbox_id, thread_id=thread_id,
    )
    _thread_registry[sender] = thread_id or ""

    if urgency == "critical" and ON_CALL_PHONE:
        _sync_client.sms.send(to=ON_CALL_PHONE, body=f"[CRITICAL] {sender}: {subject[:50]}")

@app.post("/webhook/commune")
async def webhook(request: Request, bg: BackgroundTasks):
    raw = await request.body()
    sig = request.headers.get("X-Commune-Signature", "")
    ts  = request.headers.get("X-Commune-Timestamp", "")
    try:
        _sync_client.verify_signature(payload=raw, signature=sig, secret=WEBHOOK_SECRET, timestamp=ts)
    except Exception:
        raise HTTPException(status_code=401)
    payload   = json.loads(raw)
    inbox_id  = payload.get("inbox_id", SUPPORT_INBOX_ID)
    thread_id = payload.get("thread_id") or _thread_registry.get(payload.get("sender"))
    bg.add_task(_process, payload, inbox_id, thread_id)
    return JSONResponse({"status": "accepted"})

@app.get("/health")
async def health(): return {"status": "ok"}
'''

print("=== Production app.py preview ===")
print("\nStartup sequence:")
print("  [1] Singleton clients initialized")
print("  [2] Deliverability check: bounce_rate=1.2%, spam_rate=0.11% (1 alert)")
print("  [3] FastAPI listening on 0.0.0.0:8000")
print("\nInbound email flow:")
print("  POST /webhook/commune")
print("    \u2192 verify X-Commune-Signature")
print("    \u2192 return HTTP 200")
print("    \u2192 [background] search_context = client.search.threads(subject)")
print("    \u2192 [background] reply = support_chain.invoke({search_context, ...})")
print("    \u2192 [background] client.messages.send(reply, thread_id=thread_id)")
print("    \u2192 [background] if urgency==critical: client.sms.send(on_call_phone)")
print("\n✓ Script ready \u2014 save as app.py and run with uvicorn")

=== Production app.py preview ===

Startup sequence:
  [1] Singleton clients initialized
  [2] Deliverability check: bounce_rate=1.2%, spam_rate=0.11% (1 alert)
  [3] FastAPI listening on 0.0.0.0:8000

Inbound email flow:
  POST /webhook/commune
    → verify X-Commune-Signature
    → return HTTP 200
    → [background] search_context = client.search.threads(subject)
    → [background] reply = support_chain.invoke({search_context, ...})
    → [background] client.messages.send(reply, thread_id=thread_id)
    → [background] if urgency==critical: client.sms.send(on_call_phone)

✓ Script ready — save as app.py and run with uvicorn


## Production Deployment Checklist

| Component | What to verify |
|---|---|
| Client init | Singleton pattern — one `AsyncCommuneClient` per process |
| Webhook signature | `verify_signature()` called with raw bytes before `json.loads()` |
| Background tasks | Agent work runs in background — handler returns 200 in < 1s |
| Thread continuity | `thread_id` from webhook payload passed to every `messages.send()` reply |
| Search context | `client.search.threads()` called with email subject as query |
| Deliverability cron | `client.delivery.metrics()` checked every 30 min; alert if bounce > 5% or spam > 0.1% |
| Suppression audit | `client.delivery.suppressions()` exported to CRM weekly |
| SMS escalation | `client.sms.send()` triggered only on `urgency: critical` |
| Extraction schema | Set on inbox once; all inbound emails arrive pre-parsed |

## Next Steps

- **[langchain_customer_support.ipynb](./langchain_customer_support.ipynb)** — single-agent baseline with LangChain tools
- **[crewai_02_production_patterns.ipynb](./crewai_02_production_patterns.ipynb)** — CrewAI thread persistence, retry, idempotency
- **[openai_agents_02_patterns.ipynb](./openai_agents_02_patterns.ipynb)** — parallel agents, extraction schemas, injection defense
- **Commune docs:** https://commune.email/docs